# Results analysis: Evaluator-Optimizer loop + Panacea

Loads the detailed tables from `Optimizer/results.py` for one loop run (`Optimizer/loop.py`) and, optionally, Panacea's zero-shot run (`Baseline/baseline.py panacea`, same task list).

- **Systems:** `zero_shot` (Claude's attempt 0, the official Claude zero-shot result), the four arms (`control`, `bare_retry`, `reflect_fix`, `critic`) at attempts 0–3, and `panacea` (one run per task).
- **Scores:** every attempt is scored on the **hidden** tests: `hidden_passed`/`hidden_total`, the rubric level 0–4 computed from them, and pass@1 (all hidden tests pass). Public tests are shown beside them.
- **Attempt-0 cells:** A true pass, B false pass (public pass, hidden fail), C hidden-only pass, D true fail, failed (API). Only C and D go into the arms; A and B keep their attempt-0 answer at every attempt.
- MBPP and LiveCodeBench difficulty are different scales: never compare an MBPP level with a LiveCodeBench level.

Run from inside a clone of the repo. In Colab: clone, then `%cd` into `notebooks/`.

Section 1b adds confidence intervals and the gain over the control, 1c the repair / break / false-fix / regression outcomes and cost per repair, and 12 draws the charts.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
from Optimizer.results import build
from Optimizer.charts import draw_all

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)

## Pick the run
`PANACEA_RUN = None` to look at the loop alone. The CSVs are written to `<run>/analysis/`.

In [ ]:
RUN_DIR = REPO_ROOT / "data" / "results" / "2026-11-03_claude_pilot"   # a finished loop run folder
PANACEA_RUN = None                                                     # e.g. REPO_ROOT / "panacea_run.json"

tables, meta = build(RUN_DIR, PANACEA_RUN)
print(meta["model"], "| temperature", meta["temperature"], "| max_tokens", meta["max_tokens"],
      "| task list", meta["ids_file"], f"({len(meta['task_ids'])} tasks)", "| scoring_version", meta["scoring_version"])
print("Tables:", ", ".join(tables))

## 1. Main table: every system at every attempt
Hidden pass@1 (failed API runs count 0; `_excl_failed` leaves out tasks whose attempt-0 call failed), mean share of hidden tests passed, public pass@1, false passes, rubric level counts, and calls / cost at that attempt and cumulative.

In [ ]:
main = tables["main"]
main

Hidden pass@1 as attempt columns, one row per system:

In [ ]:
main.pivot(index="system", columns="attempt", values="hidden_pass@1").reindex(main["system"].unique())

Mean share of hidden tests passed (partial credit), same layout:

In [ ]:
main.pivot(index="system", columns="attempt", values="mean_hidden_test_pass_rate").reindex(main["system"].unique())

## 1b. Curves with 95% confidence intervals
Per arm and attempt, three scopes: `all` (every task whose attempt-0 call worked; `hidden_pass@1_incl_failed` counts failed ones 0), `paired` (C + D, the self-correction view), `D` (the repair targets). `ci_low`/`ci_high`: Wilson 95% interval. `gain_vs_control`: the arm minus the control on the same tasks, with a paired bootstrap 95% interval (`gain_ci_low`/`gain_ci_high`). An interval that includes 0 means this run can't tell the arm from the control.

In [ ]:
curves = tables["curves"]
curves

In [ ]:
# hidden pass@1 [95% CI], one row per arm, attempts as columns
SCOPE = "paired"   # "all", "paired" or "D"
c = curves[curves["scope"] == SCOPE]
c = c.assign(cell=c.apply(lambda r: f"{r['hidden_pass@1']:.0%} [{r['ci_low']:.0%}, {r['ci_high']:.0%}]", axis=1))
c.pivot(index="arm", columns="attempt", values="cell").reindex(c["arm"].unique())

## 1c. Self-correction outcomes per arm (C + D, at the last attempt)
Repairs (D tasks now passing the hidden tests, with a Wilson 95% interval), breaks (C tasks now failing), false fixes (public pass, hidden fail; from D / from C and as a share of the public "fixes"), regressions (events, tasks, rate per retry that ran), failed API calls, and cost: retry calls / tokens / $, $ / tokens / calls per repair, extra $ per extra repair vs the control (negative: more repairs for less money), and the pipeline cost (attempt 0 on every task + the arm's retries).

In [ ]:
tables["arm_outcomes"].set_index("arm").T

## 2. Attempt-0 cells (Claude and Panacea)
`B_mean_hidden_test_pass_rate`: how close the false passes came to passing the hidden tests.

In [ ]:
cells = tables["cells"]
cells[[c for c in cells.columns if not c.endswith("_ids")]]

In [ ]:
cells.set_index("system")[[c for c in cells.columns if c.endswith("_ids")]].T

## 3. Rubric levels per system × attempt
0 Error/Timeout, 1 Wrong, 2 Partial Pass (< half), 3 Mostly Pass (≥ half), 4 Pass; `level_none` = failed API call.

In [ ]:
main[["system", "attempt"] + [f"level_{i}" for i in range(5)] + ["level_none"]]

## 4. Movement between attempts (tasks that went into the arms, C + D)
Each step compares with the last good attempt; a failed API call counts only in `failed_api_calls`.

In [ ]:
tables["transitions"]

## 5. What each retry buys
Per arm and attempt, over every task: hidden pass@1 gained, tasks newly solved, cost of that attempt, $ and calls per extra solved task.

In [ ]:
tables["retry_value"]

## 6. When the arms stop
Tasks that went into the arms, by the attempt of their first public-test pass.

In [ ]:
tables["stops"]

## 7. Failure types
Per system × attempt (hidden scoring), then how failure types moved from attempt 0 to attempt 3 in each arm.

In [ ]:
tables["failure_types"]

In [ ]:
tables["failure_moves"]

## 8. Breakdowns
`GROUPING`: `source_difficulty`, `test_format`, `task_type`, or `attempt0_failure_type` (Claude's attempt-0 failure type; `paired` scope only). `SCOPE`: `all` (every task, failed runs count 0) or `paired` (C + D only). `VALUE`: any numeric column of `by_group`, e.g. `hidden_pass@1`, `gain_vs_control`, `mean_hidden_test_pass_rate`.

In [ ]:
GROUPING, SCOPE, VALUE = "source_difficulty", "paired", "hidden_pass@1"
bg = tables["by_group"]
g = bg[(bg["grouping"] == GROUPING) & (bg["scope"] == SCOPE)]
g.pivot_table(index=["group", "system"], columns="attempt", values=VALUE, sort=False)

In [ ]:
g.drop_duplicates(["group"]).set_index("group")["n"].rename("tasks per group")

## 9. Paired tests (exact McNemar on per-task hidden pass/fail)
Every pair of arms at attempts 1–3, each arm against its previous attempt, each arm against zero-shot, and Panacea against Claude zero-shot and each arm at attempt 3. **Raw p values, no correction:** with this many tests, some small p values appear by chance, and with few tasks most differences can't show up at all.

In [ ]:
pt = tables["paired_tests"]
pt

In [ ]:
pt[pt["p_exact_mcnemar"] < 0.05]

## 10. Per task
One row per task: Claude's cell, hidden `passed/total` and rubric level for zero-shot, every arm at attempts 1–3, and Panacea.

In [ ]:
tasks = tables["tasks"]
tasks

In [ ]:
# e.g. the tasks that went into the arms, or one source:
tasks[tasks["claude_cell"].isin(["C_hidden_only_pass", "D_true_fail"])]

## 11. Every row
`attempts` is the long table everything above is built from (one row per system × task × attempt). Filter it for anything the tables don't show.

In [ ]:
att = tables["attempts"]
att[(att["system"] == "reflect_fix") & att["in_arms"]].sort_values(["task_id", "attempt"])

## 12. Charts
Drawn only from the tables above (`Optimizer/charts.py`; its docstring lists each chart and the table it reads). Mock and synthetic runs carry a TEST DATA banner. To save PNGs: `draw_all(tables, meta, RUN_DIR / "analysis" / "charts")`, or from the terminal `python Optimizer/results.py <run> [panacea_run.json] --charts`. `theme="dark"` for slides with a dark background.

In [ ]:
figs = draw_all(tables, meta)   # shows every chart below; figs["03_rubric_levels"] etc.